<a href="https://colab.research.google.com/github/sevensixteeeen/Pandas-lib./blob/main/non_online_storedata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mashlyn/online-retail-ii-uci")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'online-retail-ii-uci' dataset.
Path to dataset files: /kaggle/input/online-retail-ii-uci


In [2]:
import os
print(os.listdir(path))

['online_retail_II.csv']


In [3]:
import pandas as pd
import numpy as np
df=pd.read_csv(path+"/online_retail_II.csv")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


(1067371, 8)

In [5]:
df.isna().sum()
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [6]:
df = df.rename(columns={
    "Customer ID": "customer_id",
    "Price": "unit_price",
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Country": "country",
})
print(df)

        invoice stock_code                          description  quantity  \
0        489434      85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1        489434     79323P                   PINK CHERRY LIGHTS        12   
2        489434     79323W                  WHITE CHERRY LIGHTS        12   
3        489434      22041         RECORD FRAME 7" SINGLE SIZE         48   
4        489434      21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...         ...        ...                                  ...       ...   
1067366  581587      22899         CHILDREN'S APRON DOLLY GIRL          6   
1067367  581587      23254        CHILDRENS CUTLERY DOLLY GIRL          4   
1067368  581587      23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
1067369  581587      22138        BAKING SET 9 PIECE RETROSPOT          3   
1067370  581587       POST                              POSTAGE         1   

                invoice_date  unit_price  customer_id         country  
0  

In [7]:
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["customer_id"] = df["customer_id"].astype("Int64")
df["description"] = df["description"].str.strip().str.upper()
df["country"] = df["country"].str.strip()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice       1067371 non-null  object        
 1   stock_code    1067371 non-null  object        
 2   description   1062989 non-null  object        
 3   quantity      1067371 non-null  int64         
 4   invoice_date  1067371 non-null  datetime64[ns]
 5   unit_price    1067371 non-null  float64       
 6   customer_id   824364 non-null   Int64         
 7   country       1067371 non-null  object        
dtypes: Int64(1), datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 66.2+ MB


In [8]:
df["is_cancellation"] = df["invoice"].str.startswith("C")
df["is_cancellation"].sum()

np.int64(19494)

In [9]:
sales = df[
    (~df["is_cancellation"]) &
    (df["quantity"] > 0) &
    (df["unit_price"] > 0)
].copy()

print(f"raw:   {len(df):,}")
print(f"sales: {len(sales):,}  ({len(sales)/len(df):.1%} kept)")

raw:   1,067,371
sales: 1,041,670  (97.6% kept)


In [10]:
dupes = sales.duplicated().sum()
print(dupes)
sales = sales.drop_duplicates()

33757


In [11]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["year_month"] = sales["invoice_date"].dt.to_period("M")
sales["weekday"] = sales["invoice_date"].dt.day_name()
sales["hour"] = sales["invoice_date"].dt.hour

In [12]:
top_products = (
    sales.groupby("description")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
top_products

,revenue
description,
MANUAL,339241.29
REGENCY CAKESTAND 3 TIER,330590.32
DOTCOM POSTAGE,309854.11
WHITE HANGING HEART T-LIGHT HOLDER,260990.22
"PAPER CRAFT , LITTLE BIRDIE",168469.60
PARTY BUNTING,148318.28
JUMBO BAG RED RETROSPOT,148073.47
ASSORTED COLOUR BIRD ORNAMENT,129324.49
POSTAGE,125682.42


In [14]:
by_country = (
    sales.groupby("country")
    .agg(
        revenue=("revenue", "sum"),
        orders=("invoice", "nunique"),
        customers=("customer_id", "nunique"),
    )
    .sort_values("revenue", ascending=False)
)
by_country.head(10)

,revenue,orders,customers
country,,,
United Kingdom,1.741020e+07,36535,5350
EIRE,6.587673e+05,626,5
Netherlands,5.540381e+05,228,22
Germany,4.250197e+05,789,107
France,3.504561e+05,622,95
Australia,1.692835e+05,95,15
Spain,1.083325e+05,154,41
Switzerland,1.006856e+05,93,22
Sweden,9.186982e+04,105,19


In [15]:
monthly = sales.groupby("year_month")["revenue"].sum()
monthly

,revenue
year_month,
2009-12,822483.950
2010-01,651155.112
2010-02,551504.726
2010-03,830915.261
2010-04,678875.252
2010-05,657705.500
2010-06,749537.310
2010-07,648810.270
2010-08,695251.910


In [16]:
# step 1: collapse line items → invoices
invoices = (
    sales.groupby(["invoice", "country"], as_index=False)
    .agg(order_value=("revenue", "sum"))
)

# step 2: collapse invoices → countries
aov = (
    invoices.groupby("country")
    .agg(
        aov=("order_value", "mean"),
        median_order=("order_value", "median"),
        n_orders=("order_value", "size"),
    )
    .query("n_orders >= 50")
    .sort_values("aov", ascending=False)
)
aov.head(10)

,aov,median_order,n_orders
country,,,
Netherlands,2429.991623,607.19,228
Australia,1781.931158,384.68,95
Switzerland,1082.640753,535.37,93
EIRE,1052.343946,678.52,626
Sweden,874.950667,454.20,105
Channel Islands,811.333273,828.84,55
Spain,703.457727,432.89,154
Portugal,592.458211,421.54,95
France,563.434228,389.68,622
